In [1]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary, TransformInstanceLibrary, WorkflowTask
from metasmith.python_api import DataTypeLibrary, Endpoint
from local.constants import WORKSPACE_ROOT

# dtypes, containers, transforms = Std()

path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
# smith.Deploy()

2025-11-13_21-26-21  | >>> AGENT_HOME=/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
2025-11-13_21-26-21  | >>> mkdir -p $AGENT_HOME
2025-11-13_21-26-21  | >>> mkdir -p /home/tony/.globus
2025-11-13_21-26-21  | >>> mkdir -p /home/tony/.globusonline
2025-11-13_21-26-21  | >>> {if not exists}: apptainer pull/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-11-13_21-26-21  | staged [msm]
2025-11-13_21-26-21  | staged [lib/agent.yml]
2025-11-13_21-26-21  | staged [lib/msm_bootstrap]
2025-11-13_21-26-21  | staged [lib/nextflow_config]
2025-11-13_21-26-21  | deploying [4] staged files
2025-11-13_21-26-22  | >>> cd /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home && ./msm api deploy_from_container
2025-11-13_21-26-22  | including dev binds
2025-11-13_21-26-22  | binds [ --bind ./:/ws,/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home:/msm_home,/home/tony/.globus:/home/tony/.globus,/home/tony/.globusonline:/home/tony/

In [2]:
mock_types = DataTypeLibrary(types=dict(
    a=Endpoint({"test", "a"}),
    b=Endpoint({"test", "b"}),
    c=Endpoint({"test", "c"}),
))

transforms = TransformInstanceLibrary("./transforms/simple_1", include_std=False)
transforms.AddTypeLibrary("mock", mock_types)
transforms.AddStub("no_op")
transforms.Save()

In [3]:
in_path = WORKSPACE_ROOT/"main/local_mock/cache/mock_dt"
with open(in_path, "w") as f:
    f.write("120")
inputs = DataInstanceLibrary("./cache/dev20.mock.xgdb")
inputs.AddTypeLibrary("mock", mock_types)
inputs.AddItem(in_path, "mock::a")
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, p, e, e.parents)

mock::a /home/tony/workspace/tools/Metasmith/main/local_mock/cache/mock_dt <[a,test]:q7XtOGUL> set()


In [4]:
for loc, t,  in transforms.IterateTransforms():
    print(t.model)

{a-test}->{b-test}


In [5]:
N = 100
_tasks = [
    smith.GenerateWorkflow(
        given      = [inputs],
        transforms = [transforms],
        targets    = [mock_types["b"]]
    )
    # for t in ["per_contig_coverage"]
    for _ in range(N)
]
with open(WORKSPACE_ROOT/"secrets/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()
task = WorkflowTask.Merge(
    _tasks,
    config=dict(
        nextflow = dict(
            preset="slurm",
            slurm_account=SLURM_ACCOUNT,
            cpus=1,
            queueSize=100,
            array=10,
            memory='32 GB',
            time='6h',
        ),
    )
)
print(task.GetKey(), len([s for p in task.plans for s in p.steps]))
for step in [s for p in task.plans for s in p.steps][:3]:
    print(step.order, step.transform.name)
# task.RenderDAG("./cache/dag")

JwaPPFxI 100
1 no_op
2 no_op
3 no_op


In [6]:
smith.StageWorkflow(task, on_exist="clear", verify_external_paths=False)

2025-11-13_21-28-07  | connecting to deployed agent
2025-11-13_21-28-07  | starting relay service
 | > 2025-11-13_21-28-09 W| relay server already running at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/relay/XPS-laptop]
2025-11-13_21-28-09 W| task already staged at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/JwaPPFxI]
2025-11-13_21-28-09 W| clearing previously staged task
2025-11-13_21-28-09  | sending metadata for workflow [JwaPPFxI]
2025-11-13_21-28-10  | staging
2025-11-13_21-28-10  | external binds ['/home/tony/workspace/tools/Metasmith/main/local_mock']
 | > including dev binds
 | > binds [--bind /home/tony/workspace/tools/Metasmith/main/local_mock:/home/tony/workspace/tools/Metasmith/main/local_mock --bind ./:/ws,/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home:/msm_home,/home/tony/.globus:/home/tony/.globus,/home/tony/.globusonline:/home/tony/.globusonline --bind /home/tony/workspace/tools/Metasmith/

In [7]:
smith.RunWorkflow(task)

2025-11-13_21-28-13  | connecting to deployed agent
2025-11-13_21-28-13  | starting relay service
 | > 2025-11-13_21-28-14 W| relay server already running at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/relay/XPS-laptop]
2025-11-13_21-28-14  | triggering execution of [JwaPPFxI]
2025-11-13_21-28-15  | external binds ['/home/tony/workspace/tools/Metasmith/main/local_mock']
2025-11-13_21-28-15  | closing connection


In [8]:
smith.CheckWorkflow(task)

2025-11-13_21-28-15  | connecting to deployed agent
2025-11-13_21-28-15  | starting relay service
 | > 2025-11-13_21-28-16 W| relay server already running at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/relay/XPS-laptop]
 | > including dev binds
 | > binds [ --bind ./:/ws,/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home:/msm_home,/home/tony/.globus:/home/tony/.globus,/home/tony/.globusonline:/home/tony/.globusonline --bind /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/dev/metasmith:/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith,/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/dev/metasmith/bin:/app]
 | > 2025-11-13_21-28-18  | api call to [check_workflow] with [{'key': 'JwaPPFxI'}]
 | > 2025-11-13_21-28-18  | searching for logs
 | > 2025-11-13_21-28-18  | found [1] runs
 | > 2025-11-13_21-28-18  |     1: [logs.2025-11-13_21-28-15]
 | > 2025-11-13_21-28-18  | here is the mai